# Retail Demand Forecasting & Inventory Optimization
**PGDM Big Data Analytics & DBMS Project**  
Faculty: **Dr. Butchi Babu Muvva**  

> **Data note:** This is a synthetic/simulated educational dataset, not real company data. It covers 60 days only; conclusions are exploratory.

## Objective
Inspect retail sales and inventory fields, profile data quality, explore demand patterns, create a time-aware baseline forecast, and translate findings into inventory-planning recommendations.

## 1. Setup and load data
Run from repository root. The CSV is included in `data/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH = Path('../data/retail_sales_data.csv') if not Path('data/retail_sales_data.csv').exists() else Path('data/retail_sales_data.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
print('Shape:', df.shape)
display(df.head())

## 2. Data dictionary and quality checks
Check schema, missing values, duplicates, date coverage, and distinct stores/products before modelling.

In [ ]:
print('Columns:', df.columns.tolist())
display(df.dtypes.astype(str).rename('dtype').to_frame())
display(df.isna().sum().rename('missing_values').to_frame())
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Stores:', df['Store ID'].nunique(), '| Products:', df['Product ID'].nunique(), '| Cities:', df['City'].nunique())
display(df.describe(include='all').T)

## 3. Clean and validate
The following cleaning removes exact duplicates, rows missing essential keys/target, coerces numeric fields, and prevents negative sales/revenue. Keep an audit trail and do not silently remove records without reporting it.

In [ ]:
rows_before = len(df)
duplicates_before = int(df.duplicated().sum())
clean = df.drop_duplicates().dropna(subset=['Date','Store ID','Product ID','Units Sold']).copy()
for col in ['Units Sold','Revenue INR','Closing Stock','Estimated Unmet Demand']:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
clean = clean.dropna(subset=['Units Sold'])
clean['Units Sold'] = clean['Units Sold'].clip(lower=0)
clean['Revenue INR'] = clean['Revenue INR'].clip(lower=0)
print({'rows_before': rows_before, 'duplicates_removed': duplicates_before, 'rows_after': len(clean)})

## 4. Exploratory analysis and charts
Each chart is descriptive. The dataset is simulated, so patterns should not be generalized to a real retailer without validation.

In [ ]:
daily = clean.groupby('Date', as_index=False).agg(revenue=('Revenue INR','sum'), units=('Units Sold','sum'))
plt.figure(figsize=(10,4)); plt.plot(daily['Date'], daily['revenue'], marker='o', linewidth=1); plt.title('Daily Revenue Trend'); plt.xlabel('Date'); plt.ylabel('Revenue (INR)'); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
category = clean.groupby('Category', as_index=False).agg(revenue=('Revenue INR','sum')).sort_values('revenue', ascending=True)
plt.figure(figsize=(8,4)); plt.barh(category['Category'], category['revenue']); plt.title('Revenue by Category'); plt.xlabel('Revenue (INR)'); plt.tight_layout(); plt.show()

In [ ]:
top = clean.groupby('Product Name', as_index=False).agg(units=('Units Sold','sum')).nlargest(10,'units').sort_values('units')
plt.figure(figsize=(8,5)); plt.barh(top['Product Name'], top['units']); plt.title('Top 10 Products by Units Sold'); plt.xlabel('Units Sold'); plt.tight_layout(); plt.show()

In [ ]:
weekday = clean.groupby('Day of Week', as_index=False).agg(units=('Units Sold','sum'))
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday['order'] = weekday['Day of Week'].map({d:i for i,d in enumerate(weekday_order)})
weekday = weekday.sort_values('order')
plt.figure(figsize=(8,4)); plt.bar(weekday['Day of Week'], weekday['units']); plt.title('Units Sold by Day of Week'); plt.ylabel('Units'); plt.xticks(rotation=25); plt.tight_layout(); plt.show()

## 5. Baseline demand model (chronological holdout)
A Random Forest is used as a simple baseline. The latest 20% of unique dates are held out to reduce temporal leakage. This short two-month dataset is insufficient to establish annual seasonality or reliable long-term forecast accuracy.

In [ ]:
agg = (clean.groupby(['Date','Store ID','Product ID','City','Category','Promotion'], as_index=False)
       .agg(units_sold=('Units Sold','sum'), revenue_inr=('Revenue INR','sum')))
agg['day_num'] = agg['Date'].dt.day
agg['day_of_week_num'] = agg['Date'].dt.dayofweek
agg['month'] = agg['Date'].dt.month
agg['is_weekend'] = (agg['day_of_week_num'] >= 5).astype(int)
agg['promotion_flag'] = agg['Promotion'].astype(str).str.lower().eq('yes').astype(int)
features = pd.get_dummies(agg[['day_num','day_of_week_num','month','is_weekend','promotion_flag','Store ID','Product ID','Category']], drop_first=False)
y = agg['units_sold']
dates = np.sort(agg['Date'].unique()); cutoff = pd.Timestamp(dates[max(1, int(len(dates)*0.8))])
train_mask = agg['Date'] < cutoff; test_mask = ~train_mask
model = RandomForestRegressor(n_estimators=250, random_state=42, min_samples_leaf=2, n_jobs=-1)
model.fit(features.loc[train_mask], y.loc[train_mask])
pred = np.maximum(0, model.predict(features.loc[test_mask])); actual = y.loc[test_mask].to_numpy()
mae = mean_absolute_error(actual, pred); rmse = mean_squared_error(actual, pred)**0.5
nonzero = actual != 0; mape = np.mean(np.abs((actual[nonzero]-pred[nonzero])/actual[nonzero]))*100 if nonzero.any() else np.nan
metrics = pd.DataFrame([{'MAE_units':mae,'RMSE_units':rmse,'MAPE_nonzero_pct':mape,'train_rows':int(train_mask.sum()),'test_rows':int(test_mask.sum()),'train_through':str(agg.loc[train_mask,'Date'].max().date()),'test_from':str(agg.loc[test_mask,'Date'].min().date())}])
display(metrics)

## 6. Business interpretation and limitations
- Use product/store/day patterns to guide replenishment review, not automatic ordering without safeguards.
- Prioritize low-closing-stock records and positive estimated unmet demand for manual investigation.
- Compare promotions descriptively; do not claim causality because promotions are not randomized.
- Treat estimated unmet demand as a simulated estimate, not verified lost sales.
- Before operational deployment, add more history, holidays, prices, lead times, stockout labels and backtesting across multiple time windows.

## 7. Export outputs
The repository script `python/pipeline.py` exports daily aggregates, holdout predictions, metrics, data-quality summary and charts. Run it from the repository root.